# Lab 01: Welcome to the AI Sandbox

This notebook introduces your interactive workspace environment. In this lab, you will:
1. Inspect your POSIX user identity and collaborative project permissions (UID `1001`, GID `2001`, `umask 0002`).
2. Explore your scoped storage mounts (`projects/project1`, `scratch`, and read-only `datasets` & `models`).
3. Learn how to construct and verify Slurm batch job submission scripts.

In [ ]:
import os
import sys
import pwd
import grp

uid = os.getuid()
gid = os.getgid()
user_name = pwd.getpwuid(uid).pw_name
group_name = grp.getgrgid(gid).gr_name

print(f"👤 Logged in as: {user_name} (UID={uid})")
print(f"👥 Primary Team Project Group: {group_name} (GID={gid})")
print(f"📂 Working Directory: {os.getcwd()}")

assert uid >= 1000, f"Expected non-root student UID, got {uid}"
print("✅ Identity check passed!")

In [ ]:
# Verify Scoped Storage Hierarchy
expected_paths = {
    "Project Workspace": "/mnt/storage/projects/project1",
    "Ephemeral Scratch": "/mnt/storage/scratch",
    "Curated Datasets (ro)": "/mnt/storage/datasets",
    "Curated Models (ro)": "/mnt/storage/models"
}

for label, path in expected_paths.items():
    assert os.path.exists(path), f"Expected mount missing: {path}"
    print(f"✅ {label}: {path} [FOUND]")

# Verify Read-Only boundary on Datasets
try:
    with open("/mnt/storage/datasets/test_leak.txt", "w") as f:
        f.write("leak")
    sys.exit("❌ ERROR: /mnt/storage/datasets was writable!")
except OSError as e:
    assert e.errno == 30, f"Expected EROFS (errno 30), got {e}"
    print("🛡️ Datasets read-only isolation verified (kernel enforced EROFS).")

In [ ]:
# Construct a sample Slurm Batch Job Script
sbatch_script = """#!/bin/bash
#SBATCH --job-name=hello_sandbox
#SBATCH --partition=batch-cpu
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --time=00:05:00
#SBATCH --output=/mnt/storage/projects/project1/logs/%j.out

echo "Slurm batch execution successful on node $(hostname) for user $(whoami)"
"""

script_path = "/mnt/storage/projects/project1/sample_job.slurm"
with open(script_path, "w") as f:
    f.write(sbatch_script)

assert os.path.exists(script_path), "Failed to write Slurm job script"
print(f"📝 Generated Slurm batch script at: {script_path}")
print("🎉 Lab 01 Completed Successfully!")